# D — интерактивная симуляция с активностью нейронов, афферентов и силами мышц

Выберите ядро **SpikingNN (.venv)** и выполните единственную кодовую ячейку.
Это отдельная расширенная копия ячейки из `explore_genome.ipynb`.
Слева **A** — стержень, мышцы и силы; справа **B** — целевой и фактический углы,
**C** — мембранные потенциалы N0/N1 или растр их спайков, **D** — сумма
откликов Ia + II + Ib отдельно для сгибателя и разгибателя, **E** — силы
натяжения мышц в ньютонах (сгибатель — красный, разгибатель — синий).
Четыре временных графика справа занимают ту же общую высоту.
Все они используют общее скользящее окно `WINDOW_S` (сейчас **0,5 с**).

Кнопка «Нейроны» переключает вид без перезапуска модели. Ползунок, старт/стоп,
сброс и снимок работают как в исходной тетрадке. Снимок сохраняет все панели
в `results/live_activity_snapshots/`. Шрифты и размеры задаются в начале ячейки.

Потенциалы, афференты и силы мышц записываются на каждом шаге **0,1 мс**;
на их графиках используется отображение минимумов/максимумов временных интервалов,
чтобы сохранить пики при ограничении числа отрисовываемых точек. Растр использует
все времена спайков. Афференты показаны в единицах модели, а не в Гц; сумма
Ia + II + Ib — визуальный показатель, не взвешенный ток обратной связи.

Цвет нейрона соответствует его наиболее сильной связи в матрице `P`: сгибатель — красный, разгибатель — синий. Цвет одинаков в режимах потенциалов и растра; он не означает отсутствия связей с другой мышцей.

**English.** Run the single code cell in the **SpikingNN (.venv)** kernel.
Panel A shows the limb, muscles and forces; B shows desired/actual angles;
C switches between the two membrane potentials and an exact spike raster;
D shows the sum of Ia + II + Ib separately for each muscle; E shows muscle
tension in newtons (flexor red, extensor blue). The four time plots fit within
the same total height and share a rolling `WINDOW_S` window (currently 0.5 s).
Voltages, afferents and forces are recorded at every 0.1 ms integration step.
Curves use a min/max display envelope; all spike times are retained. Afferent
sums are in model units, not Hz or the weighted feedback current. Stop/start
preserves the physical state. PNG snapshots include all panels and are saved
under `results/live_activity_snapshots/`. Font and display settings are at the
top of the cell; the original notebook is unchanged.

Neuron colors follow the strongest output projection in `P`: flexor red, extensor blue. Voltage and raster views share these colors; cross-muscle projections remain possible.



<!-- current-training -->
**Переобучение: времена мышц ≥50 мс.** Геном `C_transfer`. Завершено тестов 14/18, строгий успех 0/18. Перезапустите ядро. [Результаты](results/README.md).

**English.** Muscle times ≥50 ms; winner `C_transfer`. Completed 14/18; strict success 0/18. Restart the kernel. [Results](results/README.en.md).


**Проверка этой серии:** резкий переход цели +45° → −45° вызвал падение через 0,5864 с. После остановки используйте «Сброс».

**English:** the +45° → −45° target step caused a fall after 0.5864 s in this study; use Reset after the simulation stops.


<!-- mirror-genome-variants -->

Три сохранённые версии: [быстрые мышцы](best_genome_fast_muscle.json), [времена ≥50 мс](best_genome_slow_muscle.json), [времена ≥50 мс и симметрия](best_genome_slow_muscle_mirrow.json). Выберите версию через `GENOME_VARIANT`; `GENOME_PATH` позволяет указать другой файл. По умолчанию выбран зеркальный геном. Повторный запуск ячейки обновляет старые загрузчики в памяти ядра. [Происхождение и параметры](GENOME_VERSIONS.md).

Three saved variants: fast muscles, ≥50 ms muscles, and ≥50 ms with exact mirror symmetry. Select a named version using `GENOME_VARIANT`, or override it with `GENOME_PATH`. The mirror genome is selected by default. Rerunning the cell refreshes stale cached codecs. [Sources and values](GENOME_VERSIONS.en.md).

In [ ]:
# LIVE_ACTIVITY_SIMULATION: choose a solution and settings; numerical state is never auto-reloaded.
from pathlib import Path
import os
import sys
from IPython.display import display
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/SpikingNN').is_dir())
sys.path.insert(0, str(REPO))
from scripts.AfferentBalance2026.experiments.core.config import project_root
from scripts.AfferentBalance2026.experiments.notebook_runtime import select_solution, controller_input
from scripts.AfferentBalance2026.experiments.notebook_ui import LivePendulumActivitySimulation, DEFAULT_SETTINGS

# ---------------- USER SETTINGS: change these and rerun this cell ----------------
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None              # Optional explicit path; overrides GENOME_VARIANT.
INITIAL_ANGLE_DEG = 0.0         # Actual initial angle relative to the unstable vertical.
INITIAL_SPEED_RAD_S = 0.0       # Actual initial angular velocity.
INITIAL_TARGET_DEG = 0.0        # Slider starts at this desired angle, within [-45, 45].
TARGET_STEP_DEG = 0.5           # Slider resolution; currents are evaluated by the saved polynomial.
WINDOW_S = .5                 # Shared rolling window for angles, neurons, afferents and muscle forces.
COMMAND_INTERVAL_S = 0.01       # Command blocks; neural/afferent signals are recorded every 0.1 ms.
FPS = 15                       # Maximum display refresh rate, independent of the physics step.
TIME_SCALE = 1.0                # 1.0 = one simulated second per wall-clock second.
MAX_BLOCKS_PER_FRAME = 10       # Limit catch-up work so slider/buttons remain responsive.
AUTO_START = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') != '1'  # Set False to start manually.

FONT_FAMILY = 'Times New Roman' # Font used for all labels inside the figure.
FONT_SIZE = 14                  # Base figure font size.
TITLE_SIZE = 15                 # Panel titles.
LABEL_SIZE = 14                 # Axis labels.
TICK_SIZE = 13                  # Tick labels.
LEGEND_SIZE = 13                # Legend text.
ANNOTATION_SIZE = 11            # Angles, time and force values printed on the diagram.
WIDGET_FONT_SIZE = 15           # Text size of the HTML status/readout, in pixels.
FIGSIZE = (14, 9)               # Schematic on the left, four synchronized plots on the right; total height is unchanged.
FRAME_DPI = 95                 # Interactive image resolution; lower this if display is slow.
SNAPSHOT_DPI = 220              # Resolution of the PNG saved by the snapshot button.
FORCE_SCALE_M_PER_N = 0.004     # Base display length per newton, before visual length limits.
GRAVITY_MIN_ARROW_M = 0.06      # Minimum visible gravity-arrow length; does not change physical mg.
MAX_ARROW_M = 0.16              # Cap display length only; printed force values remain unchanged.
SNAPSHOT_DIR = None             # None saves under D/results/live_activity_snapshots.
FLEXOR_COLOR = '#d62728'         # Flexor muscle: red.
EXTENSOR_COLOR = '#1764c0'       # Extensor muscle: blue.
TARGET_COLOR = '#d62728'         # Desired direction and desired-angle curve.
LIMB_COLOR = '#000000'           # Actual limb and actual-angle curve.
# Neural colors follow the strongest P[muscle, neuron] projection in the loaded genome:
# flexor-dominant neurons are red; extensor-dominant neurons are blue.
NEURON_VIEW = 'membrane'         # Initial view: 'membrane' (mV) or 'spikes' (event raster).
MAX_DISPLAY_POINTS = 2400        # Min/max display envelope per curve; full samples remain in memory.


SMOKE = os.environ.get('SPIKINGNN_NOTEBOOK_SMOKE') == '1'
EXPERIMENT = project_root()/'experiments/D_trajectory'
solution = select_solution(EXPERIMENT, GENOME_PATH, variant=GENOME_VARIANT, smoke=SMOKE)
protocol = {'initial_angle_deg': INITIAL_ANGLE_DEG, 'initial_speed_rad_s': INITIAL_SPEED_RAD_S}
recording_stride = 1  # Activity records every physical 0.1 ms step.
settings = {name: globals()[name] for name in DEFAULT_SETTINGS if name in globals()}
_previous = globals().get('live_pendulum_activity')
if _previous is not None:
    _previous.close()
live_pendulum_activity = LivePendulumActivitySimulation(
    controller_input(solution), SNAPSHOT_DIR or EXPERIMENT/'results/live_activity_snapshots',
    settings=settings, activity=True)
display(live_pendulum_activity.ui)
if AUTO_START and not SMOKE:
    live_pendulum_activity.start()
# Explicit controls: .stop(), .start(), .save_snapshot(); no training is performed.


Очень странные колебания, которые возникат изза пачечного режима...


**Рефакторинг:** общий UI находится в `../notebook_ui.py`; настройки остались в ячейке. Старые outputs и заметки сохранены. После изменения численного кода перезапустите ядро.

**English:** the shared UI lives in `../notebook_ui.py`; settings remain in the cell. Existing outputs and notes are preserved. Restart the kernel after numerical changes.
